In [ ]:
import pandas as pd 
import polars as pl 
from pathlib import Path
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace, current_run
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc

from indicator_helper_functions import (
    format_time_period_mensuel, 
    calculer_indicateur_lvl5, 
    aggreger_reporting_lvl5, 
    sommer_indicateur_lvl5,
    run_indicators_nd_aggregations_lvl5
)
from pnlp_utils import log_msg, push_dataframe_to_db, check_partition_exist, upsert_partition, create_partition_indexes

In [ ]:
# Parameters
YEAR = 2026

### Indicators definitions (Proportions)

In [ ]:
# Definitions
proportion_indicator_definitions = {
    "Cas_Suspects_Teste": {
        "numerateur": ["uIqmxJcTuub", "CIzQAR8IWH1", "sRbXNrdKvyl"],
        "denominateur": ["aZwnLALknnj", "D3h3Qvl0332"]
    },
    "Positivite_TDR": {
        "numerateur" : ["SpmQSLRPMl4", "IPnqGkUUN6U"],
        "denominateur" : ["uIqmxJcTuub", "CIzQAR8IWH1"]    
    },
    "Positivite_TDR_GE" : {
        "numerateur": ["SpmQSLRPMl4", "IPnqGkUUN6U", 'e6qMP9fVtG9'],
        "denominateur": ['uIqmxJcTuub', 'CIzQAR8IWH1', 'sRbXNrdKvyl']
    },
    "Cas_Confirme_Traité": {
        "numerateur": ['BCcFZzgTlxQ', 'CGZbvJchfjk', 'nRm30I4w9En'],
        "denominateur": ['SpmQSLRPMl4', 'IPnqGkUUN6U', 'e6qMP9fVtG9']        
    }, 
    "FE_Enceinte_MILD": {
        "numerateur": ['uV53nh3MrYl'],
        "denominateur": ['kGmdIbd3mbn']    
    },
    "Letalite_Hospi": {
        "numerateur": ['wfmDVt6RVm2', 'nFjuJLLeejV'],
        "denominateur": ['AxJhIi7tUam']        
    },
    "MILD_CPS": {
        "numerateur": ['QafSfDEL1Ku'],
        "denominateur": ['cTLKwfG8pSv'] # As from 2026 we use: "pak21wvkWJC"
    },
    "Proportion_SP2": {
        "numerateur": ['fsBcCCKHGUx'],
        "denominateur": ['kGmdIbd3mbn']
    },
    "Proportion_SP3": {
        "numerateur": ['pOZcn5GNnRD'],
        "denominateur": ['kGmdIbd3mbn']    
    }
}

proportion_indicator_mort_definitions = {
    "Mortalite_Prop": {
        "numerateur": ['wfmDVt6RVm2', 'nFjuJLLeejV'],
        "denominateur": ['fvlFcxuGRng']
    }
}  

# Reporting rates 
service_base_uid = 'pMbC0FJPkcm'
service_second_uid = 'maDtHIFrSHx'
service_hospital_uid = 'OeWrFwkFMvf'
reporting_type_comp = 'REPORTING_RATE'
reporting_type_promp = 'REPORTING_RATE_ON_TIME'

# comp-prompt
reporting_indicator_definitions = {
    "DHIS_comp_service_base": {
        "reporting_type": reporting_type_comp,
        "reporting_uid": service_base_uid
    },
    "DHIS_comp_service_second": {
        "reporting_type": reporting_type_comp,
        "reporting_uid": service_second_uid, 
    },
    "DHIS_comp_service_hosp": {
        "reporting_type": reporting_type_comp, 
        "reporting_uid": service_hospital_uid
    },
    "DHIS_prompt_service_base": {
        "reporting_type": reporting_type_promp, 
        "reporting_uid": service_base_uid
    },
    "DHIS_prompt_service_second": {
        "reporting_type": reporting_type_promp, 
        "reporting_uid": service_second_uid
    },
    "DHIS_prompt_service_hosp": {
        "reporting_type": reporting_type_promp,
        "reporting_uid": service_hospital_uid,   
    }
}

### Compute indicators (Thematique)

In [ ]:
# NOTE: Uncomment this section when is requested..

# thematique_indicator_definitions = {
#     "Cas_graves_confirmés": {
#         "element_ids": ['AxJhIi7tUam']
#     },
#     "Cas_graves_traités": {
#         "element_ids": ['CGZbvJchfjk']
#     },
#     "Cas_présumés": {
#         "element_ids": ["aK0QXqm8Zxn"]
#     },
#     "Cas_présumés_traités": {
#         "element_ids": ["JXm8J8GRJxI"]
#     },
#     "Cas_simples_confirmés": {
#         "element_ids": ["rfeqp2kdOGi"]
#     },
#     "Cas_simples_traité_PN": {
#         "element_ids": ["nRm30I4w9En"]
#     },
#     "Mortal_hosp_deces": {
#         "element_ids": ["nFjuJLLeejV"]
#     },
#     "Palu_grave_deces": {
#         "element_ids": ["wfmDVt6RVm2"]
#     },
#     "TDRs_réalisés": {
#         "element_ids" : ["CIzQAR8IWH1"]
#     }
# }

In [ ]:
# NOTE: Uncomment this section when is requested..

# Compute indicators
# thematique_indicators_results = []
# for name, definition in thematique_indicator_definitions.items():
#     print(f"Computing thematique {name}..")
 
#     thema_data = sommer_indicateur_lvl5(
#                 df_indicators_mensuel,
#                 indicateur_name=name, 
#                 element_ids=definition.get("element_ids")
#             )
    
#     thematique_indicators_results.append(thema_data)

# print("Thematique indicators completed.")
# thematique_indicators = pl.concat(thematique_indicators_results)

## Aggregations for indicators_ND at level FOSA table (proportion)

This part of the script processes and aggregates the 'indicators_ND (FOSA)' dataset to prepare it for dashboard visualization in Superset.  

>The full aggregations are computed and stored in the table, at each update we only process the current year.  
>The thematic indicators also works, we only need to uncomment

### Columns and mappings

In [ ]:
# Mapping based on your Tableau groups
group_mappings = {
    # Proportion
    "Cas_Confirme_Traité": "Proportion", 
    "Cas_Suspects_Teste": "Proportion",      
    "DHIS_comp_service_base": "Proportion",
    "DHIS_comp_service_hosp": "Proportion",
    "DHIS_comp_service_second": "Proportion",
    "DHIS_prompt_service_base": "Proportion",
    "DHIS_prompt_service_hosp": "Proportion",
    "DHIS_prompt_service_second": "Proportion",
    "FE_Enceinte_MILD": "Proportion",
    "Letalite_Hospi": "Proportion",
    "MILD_CPS": "Proportion",
    "Mortalite_Prop": "Proportion",
    "Positivite_TDR": "Proportion",
    "Positivite_TDR_GE": "Proportion",
    "Proportion_SP2": "Proportion",
    "Proportion_SP3": "Proportion",
    
    # Nombre_thematique
    "Cas_graves_confirmés": "Nombre_thematique",
    "Cas_graves_traités": "Nombre_thematique",
    "Cas_présumés": "Nombre_thematique",
    "Cas_présumés_traités": "Nombre_thematique",
    "Cas_simples_confirmés": "Nombre_thematique",
    "Cas_simples_traité_PN": "Nombre_thematique",
    "Mortal_hosp_deces": "Nombre_thematique",
    "Palu_grave_deces": "Nombre_thematique",
    "TDRs_réalisés": "Nombre_thematique",
    
    # Incidence_Annuelle
    "Incidence_Annuelle": "Incidence_Annuelle",    
    "Taux_Mortalite": "Incidence_Annuelle",
    
    # Total_Cas_Palu
    "Total_Cas_Palu": "Total_Cas_Palu",
}

# Expected columns
columns_expected = [
    "aggregation_temporelle",
    "period", 
    "annee",
    "indicateur_name",
    "province", 
    "zone_de_sante", 
    "aire_de_sante",
    "fosa",
    "fosa_dhis2_id",
    "absolute_indicateur_value", 
    "taux_indicateur_value",
    "indicateur_num", 
    "indicateur_den",
    "indicateur_group"
]

### Start DB push

In [ ]:
# table names
target_table_proportion = "indicators_ND_agg_proportion_fosa"
# target_table_thematique = "indicators_ND_agg_nom_thematique_fosa"
 
# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# We can run for a selection of years if needed..
year_targets = [YEAR] 
# year_targets = [2021, 2022, 2023, 2024, 2025]
log_msg(f"Running for year(s): {year_targets}")

In [ ]:
# Org units metadata load
ou_groups =  pl.read_csv("~/workspace/pnlp-tdb-pipeline/data/metadata/organisation_unit_groups.csv")

In [ ]:
for year_target in year_targets:
    
    log_msg(f"***Running for year: {year_target}")
    # Load data
    try:
        indicators_path = Path("~/workspace/pnlp-tdb-pipeline/data/interim") / f"PNLP_indicators_{year_target}.feather"
        indicators_level_5 = pl.read_ipc(indicators_path, memory_map=False)
    except Exception as e:
        log_msg(f"Error loading {indicators_path}. No data found", "warning")
        indicators_level_5 = pl.DataFrame()
    
    try:
        mortality_path = Path("~/workspace/pnlp-tdb-pipeline/data/interim") / f"all_cause_mortality_{year_target}.feather"
        mortality_level_5 = pl.read_ipc(mortality_path, memory_map=False) 
    except Exception as e:
        log_msg(f"Error loading {mortality_path}. No data found", "warning")
        mortality_level_5 = pl.DataFrame()
    
    try:
        comp_prompt_path = Path("~/workspace/pnlp-tdb-pipeline/data/interim") / f"PNLP_DHIS_reporting_rates_{year_target}.feather"
        comp_prompt_level_5 = pl.read_ipc(comp_prompt_path, memory_map=False) 
    except Exception as e:
        log_msg(f"Error loading {comp_prompt_path}. No data found", "warning")
        comp_prompt_level_5 = pl.DataFrame()
            
    # Following original R scripts
    df_indicators_mensuel = format_time_period_mensuel(indicators_level_5, "DE")  
    df_mortality_mensuel  = format_time_period_mensuel(mortality_level_5, "IND")
    df_cp_mensuel         = format_time_period_mensuel(comp_prompt_level_5, "CP")

    # clean
    del indicators_level_5, mortality_level_5, comp_prompt_level_5
    collected = gc.collect()
    print(collected)
    
    # Compute indicators proportions..
    proportion_indicators_results = []
    for name, definition in proportion_indicator_definitions.items():
        print(f"Computing inticator proportion {name}..")

        # NOTE: This is due to the change in denominator definition
        # see: pnlp-tdb-pipeline/03a_indicator_computation_extra_indicators.ipynb
        if name == "MILD_CPS" and YEAR >= 2026:
            den = ["pak21wvkWJC"]
        else:
            den = definition.get("denominateur")
            
        ind_data = calculer_indicateur_lvl5(
                    df_indicators_mensuel,
                    indicateur_name=name, 
                    numerator=definition.get("numerateur"), 
                    denominator=den # Denominator
                )
        proportion_indicators_results.append(ind_data)
    
    # Compute mortality proportion indicator
    for name, definition in proportion_indicator_mort_definitions.items():
        print(f"Computing mortality proportion {name}..")
        df_mortality_mensuel_concat = pl.concat(
            [df_indicators_mensuel.filter(pl.col("DE_UID").is_in(["wfmDVt6RVm2", "nFjuJLLeejV"])) ,  df_mortality_mensuel]
        )
        ind_data = calculer_indicateur_lvl5(
                df_mortality_mensuel_concat,
                indicateur_name=name,
                numerator = definition.get("numerateur"),
                denominator = definition.get("denominateur")
            )
        proportion_indicators_results.append(ind_data)
    
    # Compute reporting indicators
    for name, definition in reporting_indicator_definitions.items():
        print(f"Computing reporting rate proportion {name}..")
        rr_data = aggreger_reporting_lvl5(
                    df_cp_mensuel,
                    reporting_type=definition.get("reporting_type"),
                    reporting_uid=definition.get("reporting_uid"),
                    indicateur_name=name,
                    groups=ou_groups
                )
        proportion_indicators_results.append(rr_data)
     
    log_msg("Proportion indicators completed.")
    proportion_indicators = pl.concat(proportion_indicators_results)
    
    df_year_agg = run_indicators_nd_aggregations_lvl5(proportion_indicators, group_mappings)
    assert df_year_agg.height == df_year_agg.unique().height, "Duplicated values found!" # quick check!
    
    print("Check expected columns..", end=" ")
    df_year_agg = df_year_agg.select(columns_expected)
    print("OK!")
        
    ################ Proportions PUSH
    # Select target group 
    df_year_agg_proportion = (
        df_year_agg
        .filter(pl.col("indicateur_group") == "Proportion")
        .drop(["indicateur_group"])
    )
    
    # clean
    del df_indicators_mensuel, df_mortality_mensuel, df_mortality_mensuel_concat, df_cp_mensuel, ind_data, rr_data, proportion_indicators, proportion_indicators_results, df_year_agg
    collected = gc.collect()
    print(collected)
    
    log_msg(f"Pushing proportion FOSA aggregated data to db. Target table: {target_table_proportion}")
    tmp_table_proportions = f"{target_table_proportion}_tmp" 
    push_dataframe_to_db(
        df=df_year_agg_proportion, 
        table_name=tmp_table_proportions, 
        engine=engine, 
        push_behaviour="replace",
        chunksize=500000
    )
    
    # update partition
    upsert_partition(engine, target_table_proportion, tmp_table_proportions, year_target)
    
    log_msg("Creating indexes..")
    index_columns_prop = ["aggregation_temporelle", "province", "zone_de_sante", "aire_de_sante", "fosa", "period"]
    create_partition_indexes(engine, target_table_proportion, year_target, index_columns_prop)
    
    # clean
    del df_year_agg_proportion
    collected = gc.collect()
    print(collected)
 